# Algoritmo de Deutsch y *phase kickback*

Material autónomo en formato Jupyter para estudiar el algoritmo de Deutsch, el fenómeno de *phase kickback*, la construcción de oráculos reversibles y su implementación en Qiskit.

Convenciones matemáticas usadas en todo el notebook:

$\newcommand{\ket}[1]{|#1\rangle}$
$\newcommand{\bra}[1]{\langle #1|}$
$\newcommand{\braket}[2]{\langle #1|#2\rangle}$
$\newcommand{\C}{\mathbb{C}}$
$\newcommand{\oplusxor}{\oplus}$

La notación de Dirac se usa de forma consistente: $\ket{x}\ket{y}$ significa $\ket{x}\otimes\ket{y}$. En la parte matemática, el primer registro será el registro de entrada y el segundo registro será el registro auxiliar o de salida.


## 1. Introducción del tema

El algoritmo de Deutsch es uno de los ejemplos más simples donde un circuito cuántico obtiene información global sobre una función booleana usando menos consultas al oráculo que cualquier procedimiento clásico determinista exacto.

El problema no consiste en conocer los valores individuales $f(0)$ y $f(1)$. El objetivo es decidir una propiedad global:

$$
\text{¿se cumple } f(0)=f(1) \text{ o } f(0)\neq f(1)?
$$

Para lograrlo, el algoritmo combina cuatro ideas:

1. superposición en el registro de entrada;
2. reversibilidad mediante el oráculo unitario $U_f$;
3. *phase kickback*, que codifica $f(x)$ como una fase;
4. interferencia mediante una compuerta de Hadamard final.

La ventaja cuántica aparece porque la fase relativa creada por el oráculo contiene directamente la paridad

$$
f(0)\oplusxor f(1),
$$

que distingue las funciones constantes de las balanceadas.


## 2. Objetivos de aprendizaje

Al completar este notebook, deberías poder:

1. Formular el problema de Deutsch para funciones $f:\{0,1\}\to\{0,1\}$.
2. Clasificar las cuatro funciones booleanas de un bit como constantes o balanceadas.
3. Justificar por qué una consulta clásica no basta para decidir el problema con certeza.
4. Construir el oráculo reversible
   $$
   U_f\ket{x}\ket{y}=\ket{x}\ket{y\oplusxor f(x)}.
   $$
5. Demostrar paso a paso la identidad de *phase kickback*
   $$
   U_f\ket{x}\ket{-}=(-1)^{f(x)}\ket{x}\ket{-}.
   $$
6. Derivar completamente el algoritmo de Deutsch desde el estado inicial hasta la medición.
7. Implementar en Qiskit los cuatro oráculos posibles y el circuito completo.
8. Diagnosticar errores comunes: qubit auxiliar mal preparado, medición del registro equivocado, confusión entre fase global y fase relativa, y orden de bits en Qiskit.


## 3. Mapa conceptual

La estructura lógica del contenido es:

$$
\text{álgebra lineal}
\longrightarrow
\text{qubits}
\longrightarrow
\text{operadores } X,Z,H
\longrightarrow
\text{oráculos reversibles}
\longrightarrow
\text{phase kickback}
\longrightarrow
\text{interferencia}
\longrightarrow
\text{Deutsch}.
$$

La identidad central que conecta los temas es:

$$
\ket{-}=\frac{\ket{0}-\ket{1}}{\sqrt{2}},
\qquad
X\ket{-}=-\ket{-},
$$

y por ello

$$
U_f\ket{x}\ket{-}=(-1)^{f(x)}\ket{x}\ket{-}.
$$

Después de aplicar el oráculo sobre una superposición del registro de entrada, la información queda en la fase relativa:

$$
\frac{1}{\sqrt{2}}\left((-1)^{f(0)}\ket{0}+(-1)^{f(1)}\ket{1}\right).
$$


## 4. Preparación del entorno

La siguiente celda instala Qiskit y Qiskit Aer únicamente si no están disponibles. El código está escrito para ejecutarse sin cambios en JupyterLab, Anaconda y Google Colab.


In [ ]:
# Preparación del entorno de simulación.
# La instalación se ejecuta sólo si las bibliotecas no están disponibles.

import importlib
import subprocess
import sys

try:
    from qiskit import QuantumCircuit, transpile
    from qiskit_aer import AerSimulator
    from qiskit.quantum_info import Statevector, Operator
    import qiskit
except Exception:
    print("Qiskit no está disponible. Instalando qiskit y qiskit-aer...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "qiskit>=1.0", "qiskit-aer>=0.14"
    ])
    importlib.invalidate_caches()
    from qiskit import QuantumCircuit, transpile
    from qiskit_aer import AerSimulator
    from qiskit.quantum_info import Statevector, Operator
    import qiskit

import numpy as np
np.set_printoptions(precision=3, suppress=True)

print("Entorno listo.")
print("Versión de Qiskit:", qiskit.__version__)


## 5. Estados de un qubit

Un qubit puro es un vector unitario en $\C^2$:

$$
\ket{\psi}=\alpha\ket{0}+\beta\ket{1},
\qquad
\alpha,\beta\in\C,
\qquad
|\alpha|^2+|\beta|^2=1.
$$

Los vectores de la base computacional son

$$
\ket{0}=\begin{pmatrix}1\\0\end{pmatrix},
\qquad
\ket{1}=\begin{pmatrix}0\\1\end{pmatrix}.
$$

La medición en la base computacional produce $0$ con probabilidad $|\alpha|^2$ y $1$ con probabilidad $|\beta|^2$.

La condición de normalización no es decorativa: garantiza que las probabilidades sumen uno.


In [ ]:
# Vectores base y utilidades matemáticas.

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

plus = (ket0 + ket1) / np.sqrt(2)
minus = (ket0 - ket1) / np.sqrt(2)

I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)


def norma(v):
    """Calcula la norma euclidiana de un vector complejo."""
    return np.sqrt(np.vdot(v, v).real)


def tensor(*vectores):
    """Calcula el producto tensorial de varios vectores o matrices."""
    resultado = np.array([1], dtype=complex)
    for v in vectores:
        resultado = np.kron(resultado, v)
    return resultado


def imprimir_vector(nombre, v, tol=1e-10):
    """Imprime un vector omitiendo partes imaginarias numéricamente nulas."""
    limpio = []
    for z in v:
        if abs(z.imag) < tol:
            limpio.append(float(np.real(z)))
        else:
            limpio.append(z)
    print(f"{nombre} = {np.array(limpio)}")

imprimir_vector("|0>", ket0)
imprimir_vector("|1>", ket1)
imprimir_vector("|+>", plus)
imprimir_vector("|->", minus)


## 6. Ejemplo desarrollado: normalización

Considera el vector no normalizado

$$
\ket{v}=\ket{0}+i\ket{1}.
$$

Primero calculamos su norma:

$$
\lVert \ket{v}\rVert
=\sqrt{\braket{v}{v}}
=\sqrt{1^\ast 1+i^\ast i}
=\sqrt{1+(-i)i}
=\sqrt{1+1}
=\sqrt{2}.
$$

Por tanto, el estado normalizado es

$$
\ket{\psi}=\frac{1}{\sqrt{2}}\ket{0}+\frac{i}{\sqrt{2}}\ket{1}.
$$

Las probabilidades de medición son

$$
P(0)=\left|\frac{1}{\sqrt{2}}\right|^2=\frac12,
\qquad
P(1)=\left|\frac{i}{\sqrt{2}}\right|^2=\frac12.
$$


In [ ]:
# Ejemplo: normalización de |v> = |0> + i|1>.

v = ket0 + 1j * ket1
psi = v / norma(v)

imprimir_vector("v", v)
print("Norma de v:", norma(v))
imprimir_vector("psi normalizado", psi)
print("Probabilidades:", np.abs(psi)**2)
print("Suma de probabilidades:", np.sum(np.abs(psi)**2))


## 7. Producto tensorial y registros compuestos

Para dos qubits, el estado conjunto pertenece a $\C^2\otimes\C^2\cong\C^4$. Usaremos la convención matemática

$$
\ket{x}\ket{y}=\ket{x}\otimes\ket{y}.
$$

La base computacional de dos qubits es

$$
\ket{00},\quad \ket{01},\quad \ket{10},\quad \ket{11}.
$$

Por ejemplo,

$$
\ket{0}\ket{1}
=\begin{pmatrix}1\\0\end{pmatrix}\otimes
\begin{pmatrix}0\\1\end{pmatrix}
=
\begin{pmatrix}
1\begin{pmatrix}0\\1\end{pmatrix}\\
0\begin{pmatrix}0\\1\end{pmatrix}
\end{pmatrix}
=
\begin{pmatrix}0\\1\\0\\0\end{pmatrix}.
$$

En la simulación con Qiskit se indicará explícitamente qué qubit se mide para evitar ambigüedades de orden de bits.


In [ ]:
# Producto tensorial de estados base.

ket00 = tensor(ket0, ket0)
ket01 = tensor(ket0, ket1)
ket10 = tensor(ket1, ket0)
ket11 = tensor(ket1, ket1)

imprimir_vector("|0>|1>", ket01)
imprimir_vector("|1>|0>", ket10)


## 8. Fase global y fase relativa

Dos vectores que difieren sólo por una fase global representan el mismo estado físico:

$$
\ket{\psi}\sim e^{i\phi}\ket{\psi}.
$$

La razón es que las probabilidades no cambian:

$$
|e^{i\phi}\alpha|^2
=(e^{i\phi}\alpha)^\ast(e^{i\phi}\alpha)
=e^{-i\phi}\alpha^\ast e^{i\phi}\alpha
=|\alpha|^2.
$$

En cambio, una fase relativa sí puede cambiar resultados después de una interferencia. Por ejemplo,

$$
\ket{+}=\frac{\ket{0}+\ket{1}}{\sqrt{2}},
\qquad
\ket{-}=\frac{\ket{0}-\ket{1}}{\sqrt{2}}
$$

tienen las mismas probabilidades en la base computacional, pero se distinguen si aplicamos $H$ antes de medir:

$$
H\ket{+}=\ket{0},
\qquad
H\ket{-}=\ket{1}.
$$


In [ ]:
# Fase global: no cambia probabilidades.
# Fase relativa: puede cambiar el resultado después de aplicar H.

phi = np.pi / 3
psi_global = np.exp(1j * phi) * plus

print("Probabilidades de |+>:", np.abs(plus)**2)
print("Probabilidades de exp(i*pi/3)|+>:", np.abs(psi_global)**2)

imprimir_vector("H|+>", H @ plus)
imprimir_vector("H|->", H @ minus)


## 9. Operadores elementales: $X$, $Z$ y $H$

Las matrices básicas que usaremos son

$$
X=\begin{pmatrix}0&1\\1&0\end{pmatrix},
\qquad
Z=\begin{pmatrix}1&0\\0&-1\end{pmatrix},
\qquad
H=\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}.
$$

Sus acciones sobre la base computacional son:

$$
X\ket{0}=\ket{1},\quad X\ket{1}=\ket{0},
$$

$$
Z\ket{0}=\ket{0},\quad Z\ket{1}=-\ket{1},
$$

$$
H\ket{0}=\frac{\ket{0}+\ket{1}}{\sqrt{2}}=\ket{+},
\qquad
H\ket{1}=\frac{\ket{0}-\ket{1}}{\sqrt{2}}=\ket{-}.
$$

El operador $H$ convierte información de fase relativa en diferencia de amplitudes medibles.


In [ ]:
# Acción de X, Z y H sobre estados relevantes.

for nombre, matriz in [("X", X), ("Z", Z), ("H", H)]:
    print(f"\nOperador {nombre}")
    imprimir_vector(f"{nombre}|0>", matriz @ ket0)
    imprimir_vector(f"{nombre}|1>", matriz @ ket1)

print("\nEstados de Hadamard:")
imprimir_vector("H|0>", H @ ket0)
imprimir_vector("H|1>", H @ ket1)


## 10. Eigenestados de $X$ y $Z$

Un vector no nulo $\ket{v}$ es eigenestado de un operador $A$ si existe un escalar $\lambda$ tal que

$$
A\ket{v}=\lambda\ket{v}.
$$

Para $Z$:

$$
Z\ket{0}=\ket{0}=1\cdot\ket{0},
\qquad
Z\ket{1}=-\ket{1}=(-1)\cdot\ket{1}.
$$

Por tanto, $\ket{0}$ y $\ket{1}$ son eigenestados de $Z$.

Para $X$, los estados relevantes son $\ket{+}$ y $\ket{-}$. Demostremos el caso clave:

$$
\begin{aligned}
X\ket{-}
&=X\left(\frac{\ket{0}-\ket{1}}{\sqrt{2}}\right)\\
&=\frac{1}{\sqrt{2}}X\ket{0}-\frac{1}{\sqrt{2}}X\ket{1}\\
&=\frac{1}{\sqrt{2}}\ket{1}-\frac{1}{\sqrt{2}}\ket{0}\\
&=-\left(\frac{1}{\sqrt{2}}\ket{0}-\frac{1}{\sqrt{2}}\ket{1}\right)\\
&=-\ket{-}.
\end{aligned}
$$

Esta igualdad es el mecanismo algebraico que permite el *phase kickback*.


In [ ]:
# Verificación numérica de eigenestados.

imprimir_vector("X|+>", X @ plus)
imprimir_vector("|+>", plus)
print("¿X|+> = |+>?", np.allclose(X @ plus, plus))

imprimir_vector("X|->", X @ minus)
imprimir_vector("-|->", -minus)
print("¿X|-> = -|->?", np.allclose(X @ minus, -minus))

print("\nEigenestados de Z:")
print("¿Z|0> = |0>?", np.allclose(Z @ ket0, ket0))
print("¿Z|1> = -|1>?", np.allclose(Z @ ket1, -ket1))


## 11. Ejercicio guiado 1: normalización y fase observable

**Enunciado.** Considera

$$
\ket{\phi}=\frac{1}{2}\ket{0}+\frac{\sqrt{3}}{2}\ket{1},
\qquad
\ket{\chi}=\frac{1}{2}\ket{0}-\frac{\sqrt{3}}{2}\ket{1}.
$$

1. Verifica que ambos vectores están normalizados.
2. Calcula las probabilidades de medir $0$ y $1$ en la base computacional.
3. Determina si $\ket{\phi}$ y $\ket{\chi}$ son el mismo estado físico.

**Desarrollo paso a paso.**

Para $\ket{\phi}$:

$$
\lVert\ket{\phi}\rVert^2
=\left|\frac12\right|^2+\left|\frac{\sqrt3}{2}\right|^2
=\frac14+\frac34
=1.
$$

Para $\ket{\chi}$:

$$
\lVert\ket{\chi}\rVert^2
=\left|\frac12\right|^2+\left|-\frac{\sqrt3}{2}\right|^2
=\frac14+\frac34
=1.
$$

Las probabilidades en la base computacional coinciden:

$$
P(0)=\frac14,
\qquad
P(1)=\frac34.
$$

Sin embargo, no existe una fase global $e^{i\theta}$ tal que

$$
\ket{\chi}=e^{i\theta}\ket{\phi}.
$$

Si la primera amplitud debe conservarse, entonces $e^{i\theta}=1$. Pero con $e^{i\theta}=1$, la segunda amplitud no cambia de signo. Por tanto, la diferencia es una fase relativa, no una fase global.

**Interpretación.** Una medición directa en la base computacional no distingue estos dos estados, pero una operación posterior de interferencia sí puede distinguirlos.


In [ ]:
# Verificación del ejercicio guiado 1.

phi = 0.5 * ket0 + (np.sqrt(3) / 2) * ket1
chi = 0.5 * ket0 - (np.sqrt(3) / 2) * ket1

print("Norma de phi:", norma(phi))
print("Norma de chi:", norma(chi))
print("Probabilidades de phi:", np.abs(phi)**2)
print("Probabilidades de chi:", np.abs(chi)**2)

# Comparación después de Hadamard: aquí la fase relativa sí cambia las probabilidades.
print("Probabilidades después de H sobre phi:", np.abs(H @ phi)**2)
print("Probabilidades después de H sobre chi:", np.abs(H @ chi)**2)


## 12. Funciones booleanas de un bit

Una función booleana de un bit es una aplicación

$$
f:\{0,1\}\to\{0,1\}.
$$

Como el dominio tiene dos entradas, una función queda determinada por el par ordenado

$$
(f(0),f(1)).
$$

Cada valor puede ser $0$ o $1$. Por la regla del producto, hay

$$
2\cdot 2=2^2=4
$$

funciones posibles.

La clasificación usada por el algoritmo de Deutsch es:

$$
\text{constante} \iff f(0)=f(1),
$$

$$
\text{balanceada} \iff f(0)\neq f(1).
$$

Para funciones de un bit, esto equivale a calcular la paridad

$$
f(0)\oplusxor f(1).
$$

Si la paridad es $0$, la función es constante. Si la paridad es $1$, la función es balanceada.


In [ ]:
# Enumeración de todas las funciones booleanas de un bit.

funciones = {
    "00": {0: 0, 1: 0},
    "11": {0: 1, 1: 1},
    "01": {0: 0, 1: 1},
    "10": {0: 1, 1: 0},
}

for etiqueta, tabla in funciones.items():
    f0, f1 = tabla[0], tabla[1]
    paridad = f0 ^ f1
    tipo = "constante" if paridad == 0 else "balanceada"
    print(f"{etiqueta}: f(0)={f0}, f(1)={f1}, f(0) XOR f(1)={paridad} -> {tipo}")


## 13. Ejercicio guiado 2: conteo y clasificación

**Enunciado.** Determina cuántas funciones $f:\{0,1\}\to\{0,1\}$ existen y clasifícalas.

**Desarrollo paso a paso.**

El dominio tiene dos elementos: $0$ y $1$. Para cada elemento del dominio, la función puede tomar dos valores posibles: $0$ o $1$.

Para $x=0$:

$$
f(0)\in\{0,1\}.
$$

Para $x=1$:

$$
f(1)\in\{0,1\}.
$$

Por independencia de las elecciones:

$$
\#\{f:\{0,1\}\to\{0,1\}\}=2^2=4.
$$

Las cuatro funciones son:

$$
\begin{array}{c|cc|c}
\text{etiqueta} & f(0)&f(1)&\text{tipo}\\
\hline
00&0&0&\text{constante}\\
11&1&1&\text{constante}\\
01&0&1&\text{balanceada}\\
10&1&0&\text{balanceada}
\end{array}
$$

**Interpretación.** El algoritmo de Deutsch no identifica cuál de las cuatro funciones se tiene; identifica la propiedad global constante/balanceada.


## 14. Consulta clásica exacta

Supón que sólo podemos consultar valores de $f$.

Con una sola consulta, por ejemplo a $f(0)$, se obtiene un valor $b\in\{0,1\}$. Después de conocer $f(0)=b$, todavía hay dos funciones compatibles:

$$
f(1)=b
\quad\Rightarrow\quad
\text{constante},
$$

$$
f(1)=1\oplusxor b
\quad\Rightarrow\quad
\text{balanceada}.
$$

Por tanto, una sola consulta clásica no decide el problema con certeza. Se necesitan dos consultas para conocer ambos valores $f(0)$ y $f(1)$.

El circuito cuántico obtiene la paridad $f(0)\oplusxor f(1)$ con una sola llamada a $U_f$ porque consulta el oráculo en superposición y transforma fases relativas en una salida medible.


## 15. Oráculos reversibles

Una función clásica $f:\{0,1\}\to\{0,1\}$ no siempre puede implementarse directamente como

$$
\ket{x}\mapsto\ket{f(x)}
$$

mediante una operación unitaria, porque una operación unitaria debe ser reversible. Por ejemplo, si $f(0)=f(1)=0$, entonces tanto $\ket{0}$ como $\ket{1}$ se enviarían a $\ket{0}$, perdiendo información sobre la entrada.

La solución estándar es conservar la entrada y escribir el valor de la función mediante XOR en un segundo registro:

$$
U_f\ket{x}\ket{y}
=
\ket{x}\ket{y\oplusxor f(x)}.
$$

Esta transformación sí es reversible porque aplicar el mismo operador dos veces recupera el estado inicial:

$$
U_f^2\ket{x}\ket{y}
=
U_f\ket{x}\ket{y\oplusxor f(x)}
=
\ket{x}\ket{y\oplusxor f(x)\oplusxor f(x)}
=
\ket{x}\ket{y}.
$$

La igualdad final usa $a\oplusxor a=0$.


In [ ]:
# Matriz de U_f en la convención matemática |x>|y> con índice 2*x + y.

def matriz_oraculo(funcion):
    """Devuelve la matriz 4x4 de U_f para una función f:{0,1}->{0,1}."""
    U = np.zeros((4, 4), dtype=complex)
    for x in [0, 1]:
        for y in [0, 1]:
            y_salida = y ^ funcion[x]
            indice_entrada = 2 * x + y
            indice_salida = 2 * x + y_salida
            U[indice_salida, indice_entrada] = 1
    return U

for etiqueta, tabla in funciones.items():
    U = matriz_oraculo(tabla)
    print(f"\nU_f para función {etiqueta}:")
    print(U.astype(int))
    print("¿U_f es unitaria?", np.allclose(U.conj().T @ U, np.eye(4)))


## 16. Ejemplo desarrollado: oráculo para $f(x)=x$

Si $f(0)=0$ y $f(1)=1$, entonces $f(x)=x$. El oráculo satisface

$$
U_f\ket{x}\ket{y}=\ket{x}\ket{y\oplusxor x}.
$$

Calculamos las cuatro entradas base:

$$
U_f\ket{0}\ket{0}=\ket{0}\ket{0\oplusxor 0}=\ket{0}\ket{0}=\ket{00},
$$

$$
U_f\ket{0}\ket{1}=\ket{0}\ket{1\oplusxor 0}=\ket{0}\ket{1}=\ket{01},
$$

$$
U_f\ket{1}\ket{0}=\ket{1}\ket{0\oplusxor 1}=\ket{1}\ket{1}=\ket{11},
$$

$$
U_f\ket{1}\ket{1}=\ket{1}\ket{1\oplusxor 1}=\ket{1}\ket{0}=\ket{10}.
$$

Esto es exactamente una compuerta controlada-NOT con el primer qubit como control y el segundo como objetivo.


In [ ]:
# Oráculo para f(x)=x en Qiskit.
# q0 será la entrada; q1 será la salida.

def oraculo_qiskit(etiqueta):
    """Construye un circuito de dos qubits que implementa U_f para la etiqueta indicada."""
    qc = QuantumCircuit(2, name=f"U_{etiqueta}")
    if etiqueta == "00":
        # f(0)=0, f(1)=0: no se modifica el registro de salida.
        pass
    elif etiqueta == "11":
        # f(0)=1, f(1)=1: se invierte siempre el registro de salida.
        qc.x(1)
    elif etiqueta == "01":
        # f(0)=0, f(1)=1: se invierte la salida si la entrada es 1.
        qc.cx(0, 1)
    elif etiqueta == "10":
        # f(0)=1, f(1)=0: control negativo sobre q0.
        qc.x(0)
        qc.cx(0, 1)
        qc.x(0)
    else:
        raise ValueError("Etiqueta no válida. Usa '00', '11', '01' o '10'.")
    return qc

print(oraculo_qiskit("01").draw(output="text"))


In [ ]:
# Verificación de tablas de verdad de los oráculos Qiskit.
# Qiskit usa q0 como bit menos significativo en el vector de estado;
# por eso extraemos q0 y q1 mediante operaciones bit a bit.

def tabla_desde_circuito(oraculo):
    """Devuelve la tabla (x,y)->(x_salida,y_salida) de un circuito de oráculo."""
    filas = []
    for x in [0, 1]:
        for y in [0, 1]:
            qc = QuantumCircuit(2)
            if x == 1:
                qc.x(0)
            if y == 1:
                qc.x(1)
            qc.compose(oraculo, qubits=[0, 1], inplace=True)
            sv = Statevector.from_instruction(qc)
            indice = int(np.argmax(np.abs(sv.data) ** 2))
            x_salida = indice & 1
            y_salida = (indice >> 1) & 1
            filas.append((x, y, x_salida, y_salida))
    return filas

for etiqueta in ["00", "11", "01", "10"]:
    print(f"\nTabla de verdad del oráculo {etiqueta}")
    for fila in tabla_desde_circuito(oraculo_qiskit(etiqueta)):
        x, y, xs, ys = fila
        print(f"|{x}{y}> -> |{xs}{ys}>")


## 17. Ejercicio guiado 3: construir una tabla de $U_f$

**Enunciado.** Construye la tabla completa de $U_f$ para la función

$$
f(0)=1,
\qquad
f(1)=0.
$$

**Desarrollo paso a paso.**

El oráculo está definido por

$$
U_f\ket{x}\ket{y}=\ket{x}\ket{y\oplusxor f(x)}.
$$

Entrada $\ket{0}\ket{0}$:

$$
U_f\ket{0}\ket{0}
=\ket{0}\ket{0\oplusxor f(0)}
=\ket{0}\ket{0\oplusxor 1}
=\ket{0}\ket{1}
=\ket{01}.
$$

Entrada $\ket{0}\ket{1}$:

$$
U_f\ket{0}\ket{1}
=\ket{0}\ket{1\oplusxor 1}
=\ket{0}\ket{0}
=\ket{00}.
$$

Entrada $\ket{1}\ket{0}$:

$$
U_f\ket{1}\ket{0}
=\ket{1}\ket{0\oplusxor f(1)}
=\ket{1}\ket{0\oplusxor 0}
=\ket{1}\ket{0}
=\ket{10}.
$$

Entrada $\ket{1}\ket{1}$:

$$
U_f\ket{1}\ket{1}
=\ket{1}\ket{1\oplusxor 0}
=\ket{1}\ket{1}
=\ket{11}.
$$

**Interpretación.** La salida se invierte sólo cuando la entrada es $x=0$. En un circuito, esto puede implementarse como un control negativo: aplicar $X$ al control, luego $CX$, y finalmente restaurar el control con otro $X$.


## 18. *Phase kickback*: motivación

El oráculo $U_f$ parece escribir la información de $f(x)$ en el segundo registro:

$$
\ket{x}\ket{y}\mapsto\ket{x}\ket{y\oplusxor f(x)}.
$$

Sin embargo, si el segundo registro se prepara en el eigenestado

$$
\ket{-}=\frac{\ket{0}-\ket{1}}{\sqrt{2}},
$$

entonces la acción sobre el segundo registro regresa como una fase multiplicando al primer registro:

$$
U_f\ket{x}\ket{-}=(-1)^{f(x)}\ket{x}\ket{-}.
$$

Este fenómeno se llama *phase kickback*. Es fundamental porque permite convertir el valor de una función en una fase relativa sin medir el registro auxiliar.


## 19. Demostración completa de *phase kickback*

Partimos de

$$
\ket{-}=\frac{\ket{0}-\ket{1}}{\sqrt{2}}.
$$

Aplicamos $U_f$ por linealidad:

$$
\begin{aligned}
U_f\ket{x}\ket{-}
&=U_f\ket{x}\left(\frac{\ket{0}-\ket{1}}{\sqrt{2}}\right)\\
&=U_f\left(\frac{1}{\sqrt{2}}\ket{x}\ket{0}-\frac{1}{\sqrt{2}}\ket{x}\ket{1}\right)\\
&=\frac{1}{\sqrt{2}}U_f\ket{x}\ket{0}-\frac{1}{\sqrt{2}}U_f\ket{x}\ket{1}\\
&=\frac{1}{\sqrt{2}}\ket{x}\ket{0\oplusxor f(x)}-
\frac{1}{\sqrt{2}}\ket{x}\ket{1\oplusxor f(x)}.
\end{aligned}
$$

Ahora se separan dos casos.

**Caso 1:** $f(x)=0$.

$$
\begin{aligned}
U_f\ket{x}\ket{-}
&=\frac{1}{\sqrt{2}}\ket{x}\ket{0\oplusxor 0}
-\frac{1}{\sqrt{2}}\ket{x}\ket{1\oplusxor 0}\\
&=\frac{1}{\sqrt{2}}\ket{x}\ket{0}
-\frac{1}{\sqrt{2}}\ket{x}\ket{1}\\
&=\ket{x}\frac{\ket{0}-\ket{1}}{\sqrt{2}}\\
&=\ket{x}\ket{-}\\
&=(-1)^0\ket{x}\ket{-}.
\end{aligned}
$$

**Caso 2:** $f(x)=1$.

$$
\begin{aligned}
U_f\ket{x}\ket{-}
&=\frac{1}{\sqrt{2}}\ket{x}\ket{0\oplusxor 1}
-\frac{1}{\sqrt{2}}\ket{x}\ket{1\oplusxor 1}\\
&=\frac{1}{\sqrt{2}}\ket{x}\ket{1}
-\frac{1}{\sqrt{2}}\ket{x}\ket{0}\\
&=-\ket{x}\left(\frac{\ket{0}-\ket{1}}{\sqrt{2}}\right)\\
&=-\ket{x}\ket{-}\\
&=(-1)^1\ket{x}\ket{-}.
\end{aligned}
$$

Por tanto, para ambos casos:

$$
U_f\ket{x}\ket{-}=(-1)^{f(x)}\ket{x}\ket{-}.
$$


In [ ]:
# Verificación matricial de phase kickback para las cuatro funciones.

ket_minus_aux = minus

for etiqueta, tabla in funciones.items():
    U = matriz_oraculo(tabla)
    print(f"\nFunción {etiqueta}")
    for x, ketx in [(0, ket0), (1, ket1)]:
        entrada = tensor(ketx, ket_minus_aux)
        salida = U @ entrada
        esperado = ((-1) ** tabla[x]) * entrada
        print(f"x={x}, f(x)={tabla[x]}, ¿salida = (-1)^f(x) |x>|->?", np.allclose(salida, esperado))


## 20. Ejercicio guiado 4: *phase kickback* con registro de dos bits

**Enunciado.** Sea un registro de entrada de dos qubits y un auxiliar en $\ket{-}$. Supón que una función booleana $g:\{00,01,10,11\}\to\{0,1\}$ se representa mediante el signo

$$
s(x)=(-1)^{g(x)}.
$$

Si

$$
s(00)=-1,
\qquad
s(01)=1,
\qquad
s(10)=1,
\qquad
s(11)=-1,
$$

determina el resultado de aplicar el oráculo de fase al estado

$$
\ket{00}\ket{-}.
$$

**Desarrollo paso a paso.**

La representación por signos significa que

$$
U_g\ket{x}\ket{-}=s(x)\ket{x}\ket{-}.
$$

Para el estado dado, $x=00$. Del enunciado:

$$
s(00)=-1.
$$

Por tanto,

$$
\begin{aligned}
U_g\ket{00}\ket{-}
&=s(00)\ket{00}\ket{-}\\
&=(-1)\ket{00}\ket{-}\\
&=-\ket{00}\ket{-}.
\end{aligned}
$$

**Justificación.** No se cambia la cadena de bits $00$; sólo aparece un factor de fase. Como el estado de entrada no está en superposición con otros valores de $x$, esta fase es global para ese estado aislado. Si hubiera una superposición, el signo relativo entre ramas sí tendría consecuencias observables después de interferir.


In [ ]:
# Simulación algebraica del ejercicio guiado 4.

signos = {"00": -1, "01": 1, "10": 1, "11": -1}
estado_inicial = tensor(ket0, ket0, minus)
estado_final = signos["00"] * estado_inicial

print("El signo aplicado a |00>|-> es:", signos["00"])
print("¿estado_final = -estado_inicial?", np.allclose(estado_final, -estado_inicial))


## 21. Del *phase kickback* al algoritmo de Deutsch

El algoritmo inicia con dos registros:

$$
\ket{0}\ket{1}.
$$

Luego aplica Hadamard a ambos qubits:

$$
(H\otimes H)\ket{0}\ket{1}
=H\ket{0}\otimes H\ket{1}
=\ket{+}\ket{-}.
$$

Expandiendo el primer registro:

$$
\ket{+}\ket{-}
=\left(\frac{\ket{0}+\ket{1}}{\sqrt{2}}\right)\ket{-}
=rac{1}{\sqrt{2}}\ket{0}\ket{-}+rac{1}{\sqrt{2}}\ket{1}\ket{-}.
$$

Aplicamos el oráculo:

$$
\begin{aligned}
U_f\left(\frac{1}{\sqrt{2}}\ket{0}\ket{-}+\frac{1}{\sqrt{2}}\ket{1}\ket{-}\right)
&=rac{1}{\sqrt{2}}U_f\ket{0}\ket{-}
+\frac{1}{\sqrt{2}}U_f\ket{1}\ket{-}\\
&=\frac{1}{\sqrt{2}}(-1)^{f(0)}\ket{0}\ket{-}
+\frac{1}{\sqrt{2}}(-1)^{f(1)}\ket{1}\ket{-}\\
&=\left(\frac{(-1)^{f(0)}\ket{0}+(-1)^{f(1)}\ket{1}}{\sqrt{2}}\right)\ket{-}.
\end{aligned}
$$

El segundo registro se factoriza. Toda la información necesaria está en la fase relativa del primer qubit.


## 22. Análisis de los dos casos posibles

Después del oráculo, el primer qubit está en

$$
\ket{\varphi_f}
=rac{(-1)^{f(0)}\ket{0}+(-1)^{f(1)}\ket{1}}{\sqrt{2}}.
$$

### Caso constante

Si $f(0)=f(1)=c$, con $c\in\{0,1\}$, entonces

$$
\begin{aligned}
\ket{\varphi_f}
&=\frac{(-1)^c\ket{0}+(-1)^c\ket{1}}{\sqrt{2}}\\
&=(-1)^c\frac{\ket{0}+\ket{1}}{\sqrt{2}}\\
&=(-1)^c\ket{+}.
\end{aligned}
$$

La fase $(-1)^c$ es global. Al aplicar $H$:

$$
H\ket{\varphi_f}=(-1)^cH\ket{+}=(-1)^c\ket{0}.
$$

La medición del primer qubit produce $0$ con probabilidad $1$.

### Caso balanceado

Si $f(0)\neq f(1)$, hay dos posibilidades.

Si $(f(0),f(1))=(0,1)$:

$$
\ket{\varphi_f}
=\frac{\ket{0}-\ket{1}}{\sqrt{2}}
=\ket{-}.
$$

Si $(f(0),f(1))=(1,0)$:

$$
\ket{\varphi_f}
=\frac{-\ket{0}+\ket{1}}{\sqrt{2}}
=-\frac{\ket{0}-\ket{1}}{\sqrt{2}}
=-\ket{-}.
$$

En ambos casos, el estado es $\pm\ket{-}$. Al aplicar $H$:

$$
H(\pm\ket{-})=\pm\ket{1}.
$$

La medición del primer qubit produce $1$ con probabilidad $1$.


## 23. Fórmula general después del Hadamard final

Podemos calcular directamente las amplitudes finales. Antes del último Hadamard:

$$
\ket{\varphi_f}
=\frac{1}{\sqrt{2}}
\begin{pmatrix}
(-1)^{f(0)}\\
(-1)^{f(1)}
\end{pmatrix}.
$$

Aplicamos

$$
H=\frac{1}{\sqrt{2}}
\begin{pmatrix}
1&1\\
1&-1
\end{pmatrix}.
$$

Entonces

$$
\begin{aligned}
H\ket{\varphi_f}
&=\frac{1}{\sqrt{2}}
\begin{pmatrix}
1&1\\
1&-1
\end{pmatrix}
\frac{1}{\sqrt{2}}
\begin{pmatrix}
(-1)^{f(0)}\\
(-1)^{f(1)}
\end{pmatrix}\\
&=\frac{1}{2}
\begin{pmatrix}
1\cdot(-1)^{f(0)}+1\cdot(-1)^{f(1)}\\
1\cdot(-1)^{f(0)}+(-1)\cdot(-1)^{f(1)}
\end{pmatrix}\\
&=\frac{1}{2}
\begin{pmatrix}
(-1)^{f(0)}+(-1)^{f(1)}\\
(-1)^{f(0)}-(-1)^{f(1)}
\end{pmatrix}.
\end{aligned}
$$

Si $f(0)=f(1)$, la segunda amplitud es cero. Si $f(0)\neq f(1)$, la primera amplitud es cero. Esa cancelación es interferencia destructiva.


In [ ]:
# Simulación algebraica del algoritmo de Deutsch usando matrices NumPy.

H2 = tensor(H, H)
H_sobre_entrada = tensor(H, I)
estado_inicial_deutsch = tensor(ket0, ket1)

for etiqueta, tabla in funciones.items():
    U = matriz_oraculo(tabla)
    despues_h = H2 @ estado_inicial_deutsch
    despues_oraculo = U @ despues_h
    final = H_sobre_entrada @ despues_oraculo
    probs = np.abs(final)**2
    prob_entrada_0 = probs[0] + probs[1]  # |00>, |01>
    prob_entrada_1 = probs[2] + probs[3]  # |10>, |11>
    tipo = "constante" if tabla[0] == tabla[1] else "balanceada"
    print(f"\nFunción {etiqueta} ({tipo})")
    imprimir_vector("estado final", final)
    print("P(entrada=0):", prob_entrada_0)
    print("P(entrada=1):", prob_entrada_1)


## 24. Ejercicio guiado 5: estado del primer qubit para función balanceada

**Enunciado.** Supón que después del oráculo el estado es

$$
\left(\frac{1}{\sqrt{2}}(-1)^{f(0)}\ket{0}+\frac{1}{\sqrt{2}}(-1)^{f(1)}\ket{1}\right)\ket{-}.
$$

Si $f$ es balanceada, determina las posibilidades para el primer qubit.

**Desarrollo paso a paso.**

Si $f$ es balanceada, entonces $f(0)\neq f(1)$. Hay exactamente dos casos.

Caso A:

$$
f(0)=0,
\qquad
f(1)=1.
$$

Entonces

$$
\begin{aligned}
\ket{\varphi_f}
&=\frac{1}{\sqrt{2}}(-1)^0\ket{0}+\frac{1}{\sqrt{2}}(-1)^1\ket{1}\\
&=\frac{1}{\sqrt{2}}\ket{0}-\frac{1}{\sqrt{2}}\ket{1}\\
&=\ket{-}.
\end{aligned}
$$

Caso B:

$$
f(0)=1,
\qquad
f(1)=0.
$$

Entonces

$$
\begin{aligned}
\ket{\varphi_f}
&=\frac{1}{\sqrt{2}}(-1)^1\ket{0}+\frac{1}{\sqrt{2}}(-1)^0\ket{1}\\
&=-\frac{1}{\sqrt{2}}\ket{0}+\frac{1}{\sqrt{2}}\ket{1}\\
&=-\left(\frac{1}{\sqrt{2}}\ket{0}-\frac{1}{\sqrt{2}}\ket{1}\right)\\
&=-\ket{-}.
\end{aligned}
$$

**Interpretación.** Las únicas posibilidades son $\ket{-}$ y $-\ket{-}$. Difieren por fase global, por lo que conducen al mismo resultado de medición después del último Hadamard: el primer qubit termina como $\ket{1}$ salvo fase global.


## 25. Implementación completa en Qiskit

La implementación usa dos qubits:

- `q0`: registro de entrada;
- `q1`: registro auxiliar/salida.

El circuito sigue esta secuencia:

$$
\ket{0}\ket{0}
\xrightarrow{X\text{ en auxiliar}}
\ket{0}\ket{1}
\xrightarrow{H\otimes H}
\ket{+}\ket{-}
\xrightarrow{U_f}
\left(\frac{(-1)^{f(0)}\ket{0}+(-1)^{f(1)}\ket{1}}{\sqrt{2}}\right)\ket{-}
\xrightarrow{H\otimes I}
\begin{cases}
\pm\ket{0}\ket{-},& f\text{ constante},\\
\pm\ket{1}\ket{-},& f\text{ balanceada}.
\end{cases}
$$

Sólo medimos `q0`, porque la propiedad global queda codificada en el registro de entrada.


In [ ]:
# Circuito completo de Deutsch.

def circuito_deutsch(etiqueta, medir=True):
    """Construye el circuito de Deutsch para la función indicada por etiqueta."""
    if medir:
        qc = QuantumCircuit(2, 1)
    else:
        qc = QuantumCircuit(2)

    # Preparar |0>|1>: q0 inicia en |0>, q1 se invierte a |1>.
    qc.x(1)

    # Preparar |+>|-> mediante Hadamard en ambos qubits.
    qc.h(0)
    qc.h(1)

    # Aplicar el oráculo U_f.
    qc.compose(oraculo_qiskit(etiqueta), qubits=[0, 1], inplace=True)

    # Convertir fase relativa del primer qubit en resultado medible.
    qc.h(0)

    # Medir únicamente el qubit de entrada q0.
    if medir:
        qc.measure(0, 0)

    return qc

print(circuito_deutsch("01").draw(output="text"))


In [ ]:
# Ejecutar el algoritmo de Deutsch para las cuatro funciones.

simulador = AerSimulator()

for etiqueta, tabla in funciones.items():
    qc = circuito_deutsch(etiqueta, medir=True)
    tqc = transpile(qc, simulador)
    resultado = simulador.run(tqc, shots=1024).result()
    conteos = resultado.get_counts(tqc)
    tipo = "constante" if tabla[0] == tabla[1] else "balanceada"
    bit_esperado = "0" if tipo == "constante" else "1"

    print(f"\nFunción {etiqueta}: f(0)={tabla[0]}, f(1)={tabla[1]} -> {tipo}")
    print("Conteos:", conteos)
    print("Bit esperado:", bit_esperado)
    assert set(conteos.keys()) == {bit_esperado}


## 26. Ejercicio guiado 6: completar el oráculo para $f(0)=0, f(1)=1$

**Enunciado.** Implementa el oráculo de dos qubits para

$$
f(0)=0,
\qquad
f(1)=1.
$$

El qubit `q0` será la entrada y el qubit `q1` será la salida.

**Desarrollo paso a paso.**

El oráculo debe satisfacer

$$
U_f\ket{x}\ket{y}=\ket{x}\ket{y\oplusxor f(x)}.
$$

Como $f(x)=x$, se requiere

$$
U_f\ket{x}\ket{y}=\ket{x}\ket{y\oplusxor x}.
$$

Esto coincide con una compuerta controlada-NOT:

- si $x=0$, la salida no cambia;
- si $x=1$, la salida se invierte.

Por tanto, el código esencial es

```python
qc.cx(0, 1)
```

**Interpretación.** La compuerta `cx(0,1)` no revela directamente ambos valores de $f$; implementa la consulta reversible necesaria para que el algoritmo cuántico use interferencia.


In [ ]:
# Solución del ejercicio guiado 6.

def oraculo_fx_igual_x():
    qc = QuantumCircuit(2, name="U_f")
    qc.cx(0, 1)
    return qc

print(oraculo_fx_igual_x().draw(output="text"))
print("Tabla de verdad:")
for fila in tabla_desde_circuito(oraculo_fx_igual_x()):
    x, y, xs, ys = fila
    print(f"|{x}{y}> -> |{xs}{ys}>")


## 27. Ejercicio guiado 7: oráculo para $f(0)=1, f(1)=0$

**Enunciado.** Implementa un oráculo para

$$
f(0)=1,
\qquad
f(1)=0.
$$

**Desarrollo paso a paso.**

Queremos invertir el qubit de salida cuando el qubit de entrada vale $0$, no cuando vale $1$. Una forma de hacerlo es convertir el control negativo en control positivo:

1. aplicar $X$ al qubit de entrada: $0\leftrightarrow 1$;
2. aplicar $CX$ con ese qubit como control;
3. aplicar de nuevo $X$ al qubit de entrada para restaurarlo.

Algebraicamente, para entrada original $x$:

$$
X\ket{x}=\ket{1\oplusxor x}.
$$

La compuerta $CX$ invierte la salida si $1\oplusxor x=1$, es decir, si $x=0$. Finalmente se restaura $x$.

El circuito implementa

$$
\ket{x}\ket{y}
\mapsto
\ket{x}\ket{y\oplusxor(1\oplusxor x)}.
$$

**Interpretación.** Esta función también es balanceada; por tanto, el algoritmo de Deutsch debe producir $1$ al medir el registro de entrada.


In [ ]:
# Solución del ejercicio guiado 7.

def oraculo_fx_igual_1_xor_x():
    qc = QuantumCircuit(2, name="U_f")
    qc.x(0)
    qc.cx(0, 1)
    qc.x(0)
    return qc

print(oraculo_fx_igual_1_xor_x().draw(output="text"))
print("Tabla de verdad:")
for fila in tabla_desde_circuito(oraculo_fx_igual_1_xor_x()):
    x, y, xs, ys = fila
    print(f"|{x}{y}> -> |{xs}{ys}>")


## 28. ¿Qué ocurre si cambiamos el auxiliar?

El *phase kickback* útil depende de preparar el segundo registro en $\ket{-}$, eigenestado de $X$ con eigenvalor $-1$.

Si el auxiliar fuera $\ket{+}$, entonces

$$
X\ket{+}=\ket{+}.
$$

En ese caso, si $f(x)=1$, la inversión de salida produce eigenvalor $+1$, no un signo negativo. Por tanto,

$$
U_f\ket{x}\ket{+}=\ket{x}\ket{+}
$$

para cualquier valor de $f(x)$. El oráculo no deja una fase dependiente de $x$.

Si el auxiliar fuera $\ket{0}$, entonces

$$
U_f\ket{x}\ket{0}=\ket{x}\ket{f(x)},
$$

lo cual puede entrelazar entrada y salida cuando el registro de entrada está en superposición. En ese caso, la información no queda como una fase limpia del primer registro.

La preparación $\ket{-}$ no es arbitraria: se elige porque convierte el XOR reversible en una fase controlada.


In [ ]:
# Comparación algebraica: auxiliar |->, |+> y |0>.

def aplicar_oraculo_a_superposicion(etiqueta, auxiliar):
    tabla = funciones[etiqueta]
    U = matriz_oraculo(tabla)
    entrada = tensor(plus, auxiliar)
    return U @ entrada

for aux_nombre, aux in [("|->", minus), ("|+>", plus), ("|0>", ket0)]:
    salida = aplicar_oraculo_a_superposicion("01", aux)  # f(x)=x
    print(f"\nAuxiliar {aux_nombre}, función f(x)=x")
    imprimir_vector("salida", salida)


## 29. Ejercicio guiado 8: comparar auxiliares

**Enunciado.** Para $f(x)=x$, compara el efecto del oráculo sobre

$$
\ket{+}\ket{-},
\qquad
\ket{+}\ket{+},
\qquad
\ket{+}\ket{0}.
$$

**Desarrollo paso a paso.**

Primero, para $\ket{+}\ket{-}$:

$$
\begin{aligned}
U_f\ket{+}\ket{-}
&=U_f\left(\frac{\ket{0}+\ket{1}}{\sqrt{2}}\right)\ket{-}\\
&=\frac{1}{\sqrt{2}}U_f\ket{0}\ket{-}
+\frac{1}{\sqrt{2}}U_f\ket{1}\ket{-}\\
&=\frac{1}{\sqrt{2}}(-1)^{f(0)}\ket{0}\ket{-}
+\frac{1}{\sqrt{2}}(-1)^{f(1)}\ket{1}\ket{-}.
\end{aligned}
$$

Como $f(0)=0$ y $f(1)=1$:

$$
U_f\ket{+}\ket{-}
=\frac{\ket{0}-\ket{1}}{\sqrt{2}}\ket{-}
=\ket{-}\ket{-}.
$$

Segundo, para $\ket{+}\ket{+}$:

$$
X\ket{+}=\ket{+},
$$

así que no aparece fase negativa:

$$
U_f\ket{+}\ket{+}=\ket{+}\ket{+}.
$$

Tercero, para $\ket{+}\ket{0}$:

$$
\begin{aligned}
U_f\ket{+}\ket{0}
&=\frac{1}{\sqrt{2}}U_f\ket{0}\ket{0}
+\frac{1}{\sqrt{2}}U_f\ket{1}\ket{0}\\
&=\frac{1}{\sqrt{2}}\ket{0}\ket{0}
+\frac{1}{\sqrt{2}}\ket{1}\ket{1}.
\end{aligned}
$$

**Interpretación.** Sólo el auxiliar $\ket{-}$ produce una fase relativa directamente utilizable por el Hadamard final del algoritmo de Deutsch.


## 30. Diagnóstico de errores comunes

### Error 1: confundir fase global con fase relativa

Los estados $\ket{-}$ y $-\ket{-}$ son físicamente equivalentes si se consideran de forma aislada. Sin embargo, $\ket{+}$ y $\ket{-}$ no son equivalentes: difieren por una fase relativa entre $\ket{0}$ y $\ket{1}$.

### Error 2: medir el qubit auxiliar

El algoritmo decide la propiedad global midiendo el registro de entrada. El auxiliar termina factorizado como $\ket{-}$ salvo fases globales; no contiene la respuesta de forma directa.

### Error 3: preparar el auxiliar en $\ket{+}$

Como $X\ket{+}=\ket{+}$, la acción del XOR no produce un signo que dependa de $f(x)$.

### Error 4: leer mal el orden de bits en Qiskit

En este notebook se evita la ambigüedad midiendo sólo `q0` en un bit clásico. Cuando se miden varios qubits, es necesario documentar explícitamente la correspondencia entre qubits y bits clásicos.


## 31. Ejercicios propuestos con solución razonada

Los siguientes ejercicios consolidan las ideas anteriores. Cada uno incluye un desarrollo completo para que puedas revisar el razonamiento matemático.


### Propuesto 1: fase global

**Enunciado.** Demuestra que $\ket{\psi}$ y $-\ket{\psi}$ producen las mismas probabilidades de medición en cualquier base ortonormal $\{\ket{b_j}\}$.

**Solución.** La amplitud de obtener el resultado asociado a $\ket{b_j}$ desde $\ket{\psi}$ es

$$
a_j=\braket{b_j}{\psi}.
$$

Desde $-\ket{\psi}$, la amplitud es

$$
a'_j=\bra{b_j}(-\ket{\psi})=-\braket{b_j}{\psi}=-a_j.
$$

La probabilidad correspondiente es

$$
P'_j=|a'_j|^2=|-a_j|^2=(-a_j)^\ast(-a_j)=a_j^\ast a_j=|a_j|^2=P_j.
$$

**Interpretación.** Una fase global no altera ninguna distribución de probabilidad. Por eso $\ket{-}$ y $-\ket{-}$ son indistinguibles como estados físicos aislados.


### Propuesto 2: conteo para dos bits de entrada

**Enunciado.** ¿Cuántas funciones $f:\{0,1\}^2\to\{0,1\}$ existen?

**Solución.** El dominio $\{0,1\}^2$ contiene cuatro cadenas:

$$
00,
01,
10,
11.
$$

Para cada entrada, $f$ puede tomar dos valores. Por la regla del producto:

$$
2\cdot 2\cdot 2\cdot 2=2^4=16.
$$

Como $|\{0,1\}^2|=2^2=4$, la fórmula general para funciones $f:\{0,1\}^n\to\{0,1\}$ es

$$
2^{2^n}.
$$

Para $n=2$:

$$
2^{2^2}=2^4=16.
$$

**Interpretación.** El número de funciones crece doblemente exponencial en $n$ como conjunto de tablas de verdad. Los algoritmos oraculares buscan propiedades globales sin reconstruir toda la tabla.


### Propuesto 3: Hadamard final en forma cerrada

**Enunciado.** Sea

$$
\ket{\varphi_{a,b}}=\frac{(-1)^a\ket{0}+(-1)^b\ket{1}}{\sqrt{2}},
\qquad
a,b\in\{0,1\}.
$$

Calcula $H\ket{\varphi_{a,b}}$.

**Solución.** Escribimos el estado como vector columna:

$$
\ket{\varphi_{a,b}}
=\frac{1}{\sqrt{2}}
\begin{pmatrix}
(-1)^a\\
(-1)^b
\end{pmatrix}.
$$

Aplicamos $H$:

$$
\begin{aligned}
H\ket{\varphi_{a,b}}
&=\frac{1}{\sqrt{2}}
\begin{pmatrix}
1&1\\
1&-1
\end{pmatrix}
\frac{1}{\sqrt{2}}
\begin{pmatrix}
(-1)^a\\
(-1)^b
\end{pmatrix}\\
&=\frac12
\begin{pmatrix}
(-1)^a+(-1)^b\\
(-1)^a-(-1)^b
\end{pmatrix}.
\end{aligned}
$$

Si $a=b$, entonces $(-1)^a=(-1)^b$ y queda

$$
H\ket{\varphi_{a,b}}
=\frac12
\begin{pmatrix}
2(-1)^a\\0
\end{pmatrix}
=(-1)^a\ket{0}.
$$

Si $a\neq b$, entonces $(-1)^b=-(-1)^a$ y queda

$$
H\ket{\varphi_{a,b}}
=\frac12
\begin{pmatrix}
0\\2(-1)^a
\end{pmatrix}
=(-1)^a\ket{1}.
$$

**Interpretación.** El bit medido después del Hadamard final es $a\oplusxor b$. En Deutsch, $a=f(0)$ y $b=f(1)$.


### Propuesto 4: auxiliar en $\ket{+}$

**Enunciado.** Demuestra que, para cualquier función booleana $f$, se cumple

$$
U_f\ket{x}\ket{+}=\ket{x}\ket{+}.
$$

**Solución.** Usamos

$$
\ket{+}=\frac{\ket{0}+\ket{1}}{\sqrt{2}}.
$$

Entonces

$$
\begin{aligned}
U_f\ket{x}\ket{+}
&=U_f\ket{x}\left(\frac{\ket{0}+\ket{1}}{\sqrt{2}}\right)\\
&=\frac{1}{\sqrt{2}}\ket{x}\ket{0\oplusxor f(x)}
+\frac{1}{\sqrt{2}}\ket{x}\ket{1\oplusxor f(x)}.
\end{aligned}
$$

Si $f(x)=0$:

$$
U_f\ket{x}\ket{+}
=\frac{1}{\sqrt{2}}\ket{x}\ket{0}
+\frac{1}{\sqrt{2}}\ket{x}\ket{1}
=\ket{x}\ket{+}.
$$

Si $f(x)=1$:

$$
U_f\ket{x}\ket{+}
=\frac{1}{\sqrt{2}}\ket{x}\ket{1}
+\frac{1}{\sqrt{2}}\ket{x}\ket{0}
=\ket{x}\ket{+}.
$$

**Interpretación.** $\ket{+}$ es eigenestado de $X$ con eigenvalor $+1$, por lo que no produce el signo necesario para el algoritmo.


### Propuesto 5: implementar y verificar el oráculo $f(0)=1, f(1)=0$

**Enunciado.** Construye un circuito de Qiskit que implemente

$$
f(0)=1,
\qquad
f(1)=0,
$$

y verifica que el algoritmo de Deutsch devuelve el bit $1$.

**Solución.** El oráculo se implementa con control negativo:

$$
X_0\;CX_{0,1}\;X_0.
$$

La primera $X$ convierte el caso $x=0$ en control activo. La última $X$ restaura el valor original de entrada.


In [ ]:
# Solución del propuesto 5.

oraculo_10 = oraculo_qiskit("10")
print(oraculo_10.draw(output="text"))

qc_10 = circuito_deutsch("10", medir=True)
tqc_10 = transpile(qc_10, simulador)
conteos_10 = simulador.run(tqc_10, shots=1024).result().get_counts(tqc_10)
print("Conteos para f(0)=1, f(1)=0:", conteos_10)

assert set(conteos_10.keys()) == {"1"}


## 32. Resumen final

El algoritmo de Deutsch decide si una función $f:\{0,1\}\to\{0,1\}$ es constante o balanceada usando una sola llamada al oráculo reversible

$$
U_f\ket{x}\ket{y}=\ket{x}\ket{y\oplusxor f(x)}.
$$

La clave matemática es preparar el auxiliar en

$$
\ket{-}=\frac{\ket{0}-\ket{1}}{\sqrt{2}},
$$

porque

$$
X\ket{-}=-\ket{-}.
$$

De ahí se obtiene

$$
U_f\ket{x}\ket{-}=(-1)^{f(x)}\ket{x}\ket{-}.
$$

Al consultar en superposición:

$$
\ket{+}\ket{-}
\xrightarrow{U_f}
\left(\frac{(-1)^{f(0)}\ket{0}+(-1)^{f(1)}\ket{1}}{\sqrt{2}}\right)\ket{-}.
$$

El último Hadamard convierte la fase relativa en un resultado determinista:

$$
\begin{cases}
0, & f(0)=f(1),\\
1, & f(0)\neq f(1).
\end{cases}
$$

El resultado no proviene de leer simultáneamente dos valores clásicos, sino de explotar linealidad, reversibilidad, fase relativa e interferencia.


## 33. Referencias académicas sugeridas

- Michael A. Nielsen e Isaac L. Chuang, *Quantum Computation and Quantum Information*, Cambridge University Press.
- David Deutsch, “Quantum theory, the Church–Turing principle and the universal quantum computer”, *Proceedings of the Royal Society A*, 1985.
- IBM Quantum Learning, materiales sobre algoritmos de consulta, algoritmo de Deutsch y Deutsch–Jozsa.
- QWorld, materiales QNickel sobre *phase kickback*, algoritmo de Deutsch y algoritmos oraculares convencionales.
